# Supervised Fine-Tuning (SFT) of LLM with PyTorch FSDP and QLora on Amazon SageMaker Training jobs

## Lab 1 - Data preparation

In this notebook, we are going to prepare the dataset for later on customize Qwen/Qwen3-4B-Instruct-2507

## Prerequisites

### Install requirements

In [ ]:
%pip install -r ./scripts/requirements.txt --upgrade

### Setup and dependencies

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

***

## Visualize and upload the dataset

We are going to load [FreedomIntelligence/medical-o1-reasoning-SFT](https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT) dataset

In [ ]:
import datasets
from datasets import load_dataset

dataset = (
    load_dataset(
        "FreedomIntelligence/medical-o1-reasoning-SFT",
        "en",
        split="train",
        streaming=True,
    )
    .take(1000)
    .shuffle(buffer_size=500)
)

dataset = datasets.Dataset.from_generator(lambda: dataset, features=dataset.features)

dataset

In [ ]:
import pandas as pd

df = pd.DataFrame(dataset)

df.head()

Split the dataset into train, validation, and test

In [ ]:
from sklearn.model_selection import train_test_split

_, val = train_test_split(df, test_size=0.1, random_state=42)
train, test = train_test_split(_, test_size=0.01, random_state=42)

print("Number of train elements: ", len(train))
print("Number of validation elements: ", len(val))
print("Number of test elements: ", len(test))

Format the dataset into the messages format and then apply the model chat template

In [ ]:
from datasets import Dataset
import textwrap
from tqdm import tqdm
from transformers import AutoTokenizer

model_id = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_id)


def prepare_dataset(sample):
    messages = []

    system_text = f"""
    You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. 
    Below is an instruction that describes a task, paired with an input that provides further context. 
    Write a response that appropriately completes the request.
    Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.
    """

    system_text = textwrap.dedent(system_text).strip()
    messages.append({"role": "system", "content": system_text})
    messages.append({"role": "user", "content": sample["Question"]})
    messages.append(
        {
            "role": "assistant",
            "content": f"<think>\n{sample['Complex_CoT']}\n</think>\n\n{sample['Response']}",
        }
    )

    try:
        sample["text"] = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
        )
    except Exception as e:
        sample["text"] = None

    yield {"text": sample["text"]}


def convert_to_messages(dataset):
    """Iteratively run conversion on multi-turn conversation and flatten to messages"""
    records = []
    added = 0
    rejected = 0

    print("Original lenght: ", len(dataset))

    # Unroll your generator for every dataset row
    for row in tqdm(dataset, total=len(dataset), desc="Converting to messages"):
        for example in prepare_dataset(row):
            if example["text"] is not None and example["text"].strip() != "":
                records.append(example)
                added += 1
            else:
                rejected += 1
    print("Final length: ", len(records))
    # Convert list of dicts → Hugging Face Dataset and return
    return Dataset.from_list(records)

In [ ]:
from datasets import Dataset, DatasetDict
from random import randint

train_dataset = Dataset.from_pandas(train)
val_dataset = Dataset.from_pandas(val)
test_dataset = Dataset.from_pandas(test)

dataset = DatasetDict({"train": train_dataset, "val": val_dataset})

train_dataset = convert_to_messages(dataset["train"])

print(train_dataset[randint(0, len(train_dataset) - 1)]["text"])

val_dataset = convert_to_messages(dataset["val"])

### Upload to Amazon S3

In [ ]:
import boto3
import shutil
from sagemaker.core.helper.session_helper import Session

In [ ]:
sagemaker_session = Session()
s3_client = boto3.client("s3")

bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

In [ ]:
# save train_dataset to s3 using our SageMaker session
if default_prefix:
    input_path = f"{default_prefix}/datasets/llm-fine-tuning-sft"
else:
    input_path = f"datasets/llm-fine-tuning-sft"

train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.jsonl"
val_dataset_s3_path = f"s3://{bucket_name}/{input_path}/val/dataset.jsonl"

In [ ]:
train_dataset.to_json("./data/train/dataset.jsonl", orient="records")
val_dataset.to_json("./data/val/dataset.jsonl", orient="records")
test_dataset.to_json("./tmp/gen_qa.jsonl", orient="records")


s3_client.upload_file(
    "./data/train/dataset.jsonl", bucket_name, f"{input_path}/train/dataset.jsonl"
)
s3_client.upload_file(
    "./data/val/dataset.jsonl", bucket_name, f"{input_path}/val/dataset.jsonl"
)

shutil.rmtree("./data")

print(f"Training data uploaded to:")
print(train_dataset_s3_path)
print(val_dataset_s3_path)